# LifeMate (code baseline) — bước 1: fine-tune ASR

Fine-tune **PhoWhisper-large** trên audio train theo công thức của baseline (cả model, lr 1e-5, 3 epoch). README của baseline ghi WER dev ≈ 0.097 (bản gốc ≈ 0.25).

Chuẩn bị:
1. **Runtime → GPU A100.** Fine-tune cả model 1.55B cần ~30GB GPU nên L4/T4 không đủ. Dự kiến ~40 phút train + ~20 phút chép lời.
2. `train+public_test.zip` đã có ở `MyDrive/zalo_lifemate/` và secret `GH_TOKEN` như notebook trước.

Mọi kết quả được lưu lên Drive (`MyDrive/zalo_lifemate/models/`, `MyDrive/zalo_lifemate/asr/`). Bấm **Chạy tất cả** lần sau sẽ tự bỏ qua bước train nếu model đã có trên Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code + data

In [ ]:
import os
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/zalo_lifemate"
REPO = "/content/zalo_lifemate"
token = userdata.get("GH_TOKEN")
if not os.path.isdir(REPO):
    !git clone -q https://{token}@github.com/FirstKhoi/zalo_lifemate.git {REPO}
    !git -C {REPO} remote set-url origin https://github.com/FirstKhoi/zalo_lifemate.git
%cd {REPO}
!git pull -q https://{token}@github.com/FirstKhoi/zalo_lifemate.git main && git log --oneline -1
os.environ["PYTHONPATH"] = f"{REPO}/src"
assert os.path.isfile(f"{DRIVE}/train+public_test.zip"), f"chưa có {DRIVE}/train+public_test.zip trên Drive"
if not os.path.isdir("train+public_test"):
    !unzip -q "{DRIVE}/train+public_test.zip" -d .
!ls train+public_test

In [ ]:
!pip install -q -r requirements.txt
!python -m pytest -q tests 2>&1 | tail -2

## 2. Fine-tune PhoWhisper-large
Train trên clip của `splits/train_ids.txt`, giữ lại `splits/dev_ids.txt` (959 clip) để chấm. Log `[ft] ...` hiện dần; dòng cuối `FINAL full dev` là WER thật.

In [ ]:
ASR_LOCAL = "models/phowhisper_large_ft"
ASR_DRIVE = f"{DRIVE}/models/phowhisper_large_ft_fp16"
if os.path.isdir(ASR_DRIVE):
    print("Đã có model trên Drive, bỏ qua train:", ASR_DRIVE)
    !cat {ASR_DRIVE}/dev_metrics.json | head -12
else:
    !python scripts/finetune_asr.py --data ./train+public_test --split train --model vinai/PhoWhisper-large --out {ASR_LOCAL} --epochs 3 --batch 4 --lr 1e-5 --bf16 --eval-every 800 --eval-n 300 2>&1 | grep --line-buffered -E "^\[ft\]|Error|Traceback"
    # bản fp16 trên Drive (3GB thay vì 6GB): còn lại sau khi tắt phiên, dùng cho các bước sau
    !python -c "import torch; from transformers import WhisperForConditionalGeneration as W, WhisperProcessor as P; W.from_pretrained('{ASR_LOCAL}', dtype=torch.float16).save_pretrained('{ASR_DRIVE}'); P.from_pretrained('{ASR_LOCAL}').save_pretrained('{ASR_DRIVE}')"
    !cp {ASR_LOCAL}/dev_metrics.json {ASR_DRIVE}/ && ls {ASR_DRIVE}

## 3. WER trên dev: có và không có *domain prompt*
*Domain prompt* là câu liệt kê từ vựng (Zalo, SMS, VTV1, RON95, SJC…) đưa vào Whisper trước khi nghe. Chọn cách có `WER` thấp hơn cho bước 4.

In [ ]:
!python scripts/precompute_asr.py --data ./train+public_test --split train --backend hf --model {ASR_DRIVE} --prompt domain --gpus 1 --ids-file splits/dev_ids.txt --out out/asr_dev_domain.json 2>&1 | grep --line-buffered -E "\"WER\"|key_token|money_exact|wrote|ERROR|Error|Traceback"
!python scripts/precompute_asr.py --data ./train+public_test --split train --backend hf --model {ASR_DRIVE} --gpus 1 --ids-file splits/dev_ids.txt --out out/asr_dev_noprompt.json 2>&1 | grep --line-buffered -E "\"WER\"|key_token|money_exact|wrote|ERROR|Error|Traceback"

## 4. Chép lời public_test bằng model đã fine-tune

In [ ]:
PROMPT_ARG = "--prompt domain"  # đổi thành "" nếu ở bước 3 không dùng prompt cho WER thấp hơn
!python scripts/precompute_asr.py --data ./train+public_test --split public_test --backend hf --model {ASR_DRIVE} {PROMPT_ARG} --gpus 1 --no-metrics --out out/asr_cache_public_test.json 2>&1 | grep --line-buffered -E "\"WER\"|key_token|money_exact|wrote|ERROR|Error|Traceback"
!mkdir -p "{DRIVE}/asr" && cp out/asr_*.json "{DRIVE}/asr/" && ls -la "{DRIVE}/asr"